In [2]:
data = "../../mtoralzml/data/padel_loop_results_BBB.csv"
import pandas as pd
df = pd.read_csv(data)
df

,Original_Name,Original_SMILES,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,...,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds,BBB
0,sulphasalazine,O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O,1,-1.6366,2.678460,20.9255,52.325102,18,18,42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,moxalactam,COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...,4,-1.8532,3.434350,84.1596,65.253860,11,11,56,...,0.202821,0.515082,0.311531,0.404004,22.493883,107.951242,249.714723,0.589017,1.230617,0
2,clioquinol,Oc1c(I)cc(Cl)c2cccnc12,0,1.7041,2.903957,22.1264,28.605965,10,11,18,...,0.348308,0.488440,0.490672,0.207540,7.749546,14.229627,23.592414,0.476530,1.186652,0
3,bbcpd11 (cimetidine analog) (y-g13),CCNC(=NCCSCc1ncccc1Br)NC#N,0,1.3081,1.711126,58.1882,43.238688,6,6,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,schembl614298,CN1CC[C@]23c4c5ccc(OC6O[C@H](C(=O)O)[C@@H](O)[...,1,-2.3618,5.578099,88.3588,66.801411,6,6,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9579,licostinel,C1=C(Cl)C(=C(C2=C1NC(=O)C(N2)=O)[N+](=O)[O-])Cl,0,0.1909,0.036443,57.1443,26.948379,6,6,20,...,0.350759,0.566435,0.535897,0.365741,8.796186,19.954489,35.953677,0.424423,1.468073,1
9580,ademetionine(adenosyl-methionine),[C@H]3([N]2C1=C(C(=NC=N1)N)N=C2)[C@@H]([C@@H](...,0,-4.7530,22.591009,84.4263,54.579446,0,0,49,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
9581,mesocarb,[O+]1=N[N](C=C1[N-]C(NC2=CC=CC=C2)=O)C(CC3=CC=...,0,1.4656,2.147983,98.9391,49.686274,0,0,42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
9582,tofisoline,C1=C(OC)C(=CC2=C1C(=[N+](C(=C2CC)C)[NH-])C3=CC...,0,-0.8870,0.786769,113.2376,61.464618,0,0,54,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


In [19]:
# Generate PostgreSQL table schema (for reference)
sql_schema = """
-- PostgreSQL table schema for classified molecules
CREATE TABLE molecules (
    id SERIAL PRIMARY KEY,
    name VARCHAR(500),
    smiles TEXT,
    inchi_key VARCHAR(255),
    
    -- Physicochemical properties
    tpsa FLOAT,
    logp FLOAT,
    mw FLOAT,
    hbd INT,
    hba INT,
    rotatable_bonds INT,
    ring_count INT,
    heterocycle_present BOOLEAN,
    molar_refractivity FLOAT,
    
    -- Structural properties
    peptide_like BOOLEAN,
    lipid_like BOOLEAN,
    aromatic BOOLEAN,
    
    -- Classification bins
    polarity_bin VARCHAR(50),
    lipophilicity_bin VARCHAR(50),
    size_bin VARCHAR(50),
    
    -- Drug candidacy flags
    lipinski_pass BOOLEAN,
    veber_pass BOOLEAN,
    egan_pass BOOLEAN,
    ghose_pass BOOLEAN,
    pains_flag BOOLEAN,
    
    -- JSON profile (JSONB for efficient querying in PostgreSQL)
    profile_json JSONB,
    
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Create indexes for faster queries
CREATE INDEX idx_lipinski ON molecules(lipinski_pass);
CREATE INDEX idx_veber ON molecules(veber_pass);
CREATE INDEX idx_egan ON molecules(egan_pass);
CREATE INDEX idx_ghose ON molecules(ghose_pass);
CREATE INDEX idx_pains ON molecules(pains_flag);
CREATE INDEX idx_polarity ON molecules(polarity_bin);
CREATE INDEX idx_lipophilicity ON molecules(lipophilicity_bin);
CREATE INDEX idx_size ON molecules(size_bin);
CREATE INDEX idx_aromatic ON molecules(aromatic);
"""

print("\n=== POSTGRESQL SCHEMA ===")
print(sql_schema)

print("\n=== FILES GENERATED ===")
print("✓ classified_molecules.csv - Main data file for PostgreSQL import")
print("✓ classified_molecules.json - JSON export for backup/inspection")
print("✓ Notebook contains all classification logic and functions")

print("\n=== NEXT STEPS ===")
print("1. Load CSV into PostgreSQL using: \\COPY molecules FROM 'classified_molecules.csv' WITH (FORMAT csv, HEADER true)")
print("2. Parse profile_json column as JSONB for efficient querying")
print("3. Create indexes as shown in schema above")
print("4. Build user interface to query by:")
print("   - Polarity/Lipophilicity bins")
print("   - Drug rule compliance (Lipinski, Veber, Egan, Ghose)")
print("   - PAINS alerts")
print("   - Structural properties (aromatic, peptide-like, lipid-like)")
print("   - Molecular weight and size ranges")


=== POSTGRESQL SCHEMA ===

-- PostgreSQL table schema for classified molecules
CREATE TABLE molecules (
    id SERIAL PRIMARY KEY,
    name VARCHAR(500),
    smiles TEXT,
    inchi_key VARCHAR(255),

    -- Physicochemical properties
    tpsa FLOAT,
    logp FLOAT,
    mw FLOAT,
    hbd INT,
    hba INT,
    rotatable_bonds INT,
    ring_count INT,
    heterocycle_present BOOLEAN,
    molar_refractivity FLOAT,

    -- Structural properties
    peptide_like BOOLEAN,
    lipid_like BOOLEAN,
    aromatic BOOLEAN,

    -- Classification bins
    polarity_bin VARCHAR(50),
    lipophilicity_bin VARCHAR(50),
    size_bin VARCHAR(50),

    -- Drug candidacy flags
    lipinski_pass BOOLEAN,
    veber_pass BOOLEAN,
    egan_pass BOOLEAN,
    ghose_pass BOOLEAN,
    pains_flag BOOLEAN,

    -- JSON profile (JSONB for efficient querying in PostgreSQL)
    profile_json JSONB,

    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Create indexes for faster queries
CREATE INDEX idx_lipinski ON m

In [20]:
# ============================================
# SUPABASE SETUP AND IMPORT
# ============================================

# Install supabase client
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "supabase"])

print("Supabase client installed!")

  DEPRECATION: Building 'pyroaring' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pyroaring'. Discussion can be found at https://github.com/pypa/pip/issues/6334


Supabase client installed!


In [18]:
# Display sample of final dataset (without JSON column for readability)
display_df = final_dataset[[
    'id', 'name', 'tpsa', 'logp', 'mw', 'hbd', 'hba', 
    'polarity_bin', 'lipophilicity_bin', 'size_bin',
    'lipinski_pass', 'veber_pass', 'egan_pass', 'ghose_pass'
]].head(10)

print("\n=== SAMPLE DATA (First 10 molecules) ===")
display_df


=== SAMPLE DATA (First 10 molecules) ===


,id,name,tpsa,logp,mw,hbd,hba,polarity_bin,lipophilicity_bin,size_bin,lipinski_pass,veber_pass,egan_pass,ghose_pass
0,0,sulphasalazine,NaN,4.08157,398.068491,3,9,None,lipophilic,large,True,None,None,False
1,1,moxalactam,302.211486,-1.36323,520.101247,4,15,polar,hydrophilic,very_large,False,False,False,False
2,2,clioquinol,79.338804,3.56899,304.910439,1,2,moderate,lipophilic,large,True,True,True,False
3,3,bbcpd11 (cimetidine analog) (y-g13),NaN,2.23388,341.030979,2,5,None,moderate,large,True,None,None,True
4,4,schembl614298,NaN,-1.31031,461.168581,5,10,None,hydrophilic,large,True,None,None,False
5,5,"uk-240,455",211.845275,1.48897,366.979647,3,8,polar,moderate,large,True,False,False,True
6,6,morphine-6-glucuronide,NaN,-0.95921,461.168581,5,10,None,balanced,large,True,None,None,False
7,7,nitrofurantoin,NaN,0.56638,238.033819,1,9,None,balanced,medium,True,None,None,False
8,8,"l-701,324",136.259431,5.20443,363.066221,2,4,polar,lipophilic,large,False,True,False,False
9,9,33419-42-0,NaN,1.48159,588.184291,3,13,None,moderate,very_large,False,None,None,False


In [17]:
# Export to CSV for PostgreSQL import
output_path = 'classified_molecules.csv'
final_dataset.to_csv(output_path, index=False)
print(f"\n✓ Exported to: {output_path}")

# Also export to JSON for inspection
json_output_path = 'classified_molecules.json'
final_dataset.to_json(json_output_path, orient='records', indent=2)
print(f"✓ Exported to: {json_output_path}")

# Show summary statistics
print("\n=== CLASSIFICATION SUMMARY ===")
print(f"Total molecules: {len(final_dataset)}")
print(f"\nDrug Pass Rates:")
print(f"  Lipinski: {final_dataset['lipinski_pass'].sum()}/{final_dataset['lipinski_pass'].notna().sum()} ({100*final_dataset['lipinski_pass'].sum()/final_dataset['lipinski_pass'].notna().sum():.1f}%)")
print(f"  Veber: {final_dataset['veber_pass'].sum()}/{final_dataset['veber_pass'].notna().sum()} ({100*final_dataset['veber_pass'].sum()/final_dataset['veber_pass'].notna().sum():.1f}%)")
print(f"  Egan: {final_dataset['egan_pass'].sum()}/{final_dataset['egan_pass'].notna().sum()} ({100*final_dataset['egan_pass'].sum()/final_dataset['egan_pass'].notna().sum():.1f}%)")
print(f"  Ghose: {final_dataset['ghose_pass'].sum()}/{final_dataset['ghose_pass'].notna().sum()} ({100*final_dataset['ghose_pass'].sum()/final_dataset['ghose_pass'].notna().sum():.1f}%)")
print(f"\nPAINS Alerts:")
print(f"  Flagged: {final_dataset['pains_flag'].sum()} ({100*final_dataset['pains_flag'].sum()/len(final_dataset):.1f}%)")
print(f"\nAromaticity:")
print(f"  Aromatic: {final_dataset['aromatic'].sum()}/{final_dataset['aromatic'].notna().sum()} ({100*final_dataset['aromatic'].sum()/final_dataset['aromatic'].notna().sum():.1f}%)")


✓ Exported to: classified_molecules.csv
✓ Exported to: classified_molecules.json

=== CLASSIFICATION SUMMARY ===
Total molecules: 9584

Drug Pass Rates:
  Lipinski: 7287/9584 (76.0%)
  Veber: 4255/6998 (60.8%)
  Egan: 4033/6998 (57.6%)
  Ghose: 5259/9520 (55.2%)

PAINS Alerts:
  Flagged: 3996 (41.7%)

Aromaticity:
  Aromatic: 7240/9571 (75.6%)


In [16]:
# Show sample data
print("\n=== SAMPLE RECORD ===")
sample = final_dataset.iloc[0]
print(f"ID: {sample['id']}")
print(f"Name: {sample['name']}")
print(f"SMILES: {sample['smiles'][:80]}..." if len(str(sample['smiles'])) > 80 else f"SMILES: {sample['smiles']}")
print(f"\nPhysicochemical Properties:")
print(f"  TPSA: {sample['tpsa']:.2f}, LogP: {sample['logp']:.2f}, MW: {sample['mw']:.2f}")
print(f"  Polarity: {sample['polarity_bin']}, Lipophilicity: {sample['lipophilicity_bin']}")
print(f"\nStructural Properties:")
print(f"  Size: {sample['size_bin']}, Aromatic: {sample['aromatic']}")
print(f"  Rings: {sample['ring_count']}, Heterocycles: {sample['heterocycle_present']}")
print(f"  Peptide-like: {sample['peptide_like']}, Lipid-like: {sample['lipid_like']}")
print(f"\nDrug Candidacy:")
print(f"  Lipinski: {sample['lipinski_pass']}, Veber: {sample['veber_pass']}")
print(f"  Egan: {sample['egan_pass']}, Ghose: {sample['ghose_pass']}")
print(f"  PAINS Flag: {sample['pains_flag']}")
print(f"\nProfile JSON:")
profile = json.loads(sample['profile_json'])
print(json.dumps(profile, indent=2))


=== SAMPLE RECORD ===
ID: 0
Name: sulphasalazine
SMILES: O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O

Physicochemical Properties:
  TPSA: nan, LogP: 4.08, MW: 398.07
  Polarity: None, Lipophilicity: lipophilic

Structural Properties:
  Size: large, Aromatic: True
  Rings: 3.0, Heterocycles: True
  Peptide-like: False, Lipid-like: True

Drug Candidacy:
  Lipinski: True, Veber: None
  Egan: None, Ghose: False
  PAINS Flag: False

Profile JSON:
{
  "physicochemical": {
    "tpsa": null,
    "polarity_bin": null,
    "logp": 4.081570000000003,
    "lipophilicity_bin": "lipophilic"
  },
  "structural": {
    "size_bin": "large",
    "aromatic": true,
    "ring_count": 3,
    "heterocycle_present": true,
    "peptide_like": false,
    "lipid_like": true
  },
  "drug_candidacy": {
    "lipinski": {
      "pass": true,
      "violations": 0
    },
    "veber": {
      "pass": null
    },
    "egan": {
      "pass": null
    },
    "ghose": {
      "pass": false
    },
    "pains": {
    

In [15]:
# Build final dataset with all required columns (drop intermediate columns)
final_dataset = classification_df[[
    'id',
    'name',
    'smiles',
    'tpsa',
    'logp',
    'mw',
    'hbd',
    'hba',
    'rotatable_bonds',
    'ring_count',
    'heterocycle_present',
    'molar_refractivity',
    'peptide_like',
    'lipid_like',
    'polarity_bin',
    'lipophilicity_bin',
    'size_bin',
    'aromatic',
    'lipinski_pass',
    'veber_pass',
    'egan_pass',
    'ghose_pass',
    'pains_flag',
    'profile_json'
]].copy()

print(f"\n✓ Final dataset created!")
print(f"  Shape: {final_dataset.shape}")
print(f"  Columns: {list(final_dataset.columns)}")


✓ Final dataset created!
  Shape: (9584, 24)
  Columns: ['id', 'name', 'smiles', 'tpsa', 'logp', 'mw', 'hbd', 'hba', 'rotatable_bonds', 'ring_count', 'heterocycle_present', 'molar_refractivity', 'peptide_like', 'lipid_like', 'polarity_bin', 'lipophilicity_bin', 'size_bin', 'aromatic', 'lipinski_pass', 'veber_pass', 'egan_pass', 'ghose_pass', 'pains_flag', 'profile_json']


In [14]:
# Create JSON profile column
print("Creating JSON profiles...")

def create_profile_json(row):
    """Create comprehensive classification profile as JSON"""
    profile = {
        "physicochemical": {
            "tpsa": float(row['tpsa']) if pd.notna(row['tpsa']) else None,
            "polarity_bin": row['polarity_bin'],
            "logp": float(row['logp']) if pd.notna(row['logp']) else None,
            "lipophilicity_bin": row['lipophilicity_bin']
        },
        "structural": {
            "size_bin": row['size_bin'],
            "aromatic": bool(row['aromatic']) if pd.notna(row['aromatic']) else None,
            "ring_count": int(row['ring_count']) if pd.notna(row['ring_count']) else None,
            "heterocycle_present": bool(row['heterocycle_present']) if pd.notna(row['heterocycle_present']) else None,
            "peptide_like": bool(row['peptide_like']) if pd.notna(row['peptide_like']) else None,
            "lipid_like": bool(row['lipid_like']) if pd.notna(row['lipid_like']) else None
        },
        "drug_candidacy": {
            "lipinski": {
                "pass": bool(row['lipinski_pass']) if pd.notna(row['lipinski_pass']) else None,
                "violations": int(row['lipinski_violations']) if pd.notna(row['lipinski_violations']) else None
            },
            "veber": {
                "pass": bool(row['veber_pass']) if pd.notna(row['veber_pass']) else None
            },
            "egan": {
                "pass": bool(row['egan_pass']) if pd.notna(row['egan_pass']) else None
            },
            "ghose": {
                "pass": bool(row['ghose_pass']) if pd.notna(row['ghose_pass']) else None
            },
            "pains": {
                "flag": bool(row['pains_flag']) if pd.notna(row['pains_flag']) else None,
                "alerts": row['pains_alerts'] if isinstance(row['pains_alerts'], list) else []
            }
        }
    }
    return json.dumps(profile)

classification_df['profile_json'] = classification_df.apply(create_profile_json, axis=1)

print("  ✓ JSON profiles created")

Creating JSON profiles...
  ✓ JSON profiles created


In [13]:
# Calculate drug candidacy properties
print("Calculating drug candidacy properties...")

# Lipinski's Rule of Five
lipinski_results = classification_df.apply(
    lambda row: check_lipinski(row['mw'], row['logp'], row['hbd'], row['hba']),
    axis=1
)
classification_df['lipinski_pass'] = lipinski_results.apply(lambda x: x[0])
classification_df['lipinski_violations'] = lipinski_results.apply(lambda x: x[1])

# Veber's Rule
classification_df['veber_pass'] = classification_df.apply(
    lambda row: check_veber(row['tpsa'], row['rotatable_bonds']),
    axis=1
)

# Egan Rule
classification_df['egan_pass'] = classification_df.apply(
    lambda row: check_egan(row['logp'], row['tpsa']),
    axis=1
)

# Ghose Filter
classification_df['ghose_pass'] = classification_df.apply(
    lambda row: check_ghose(row['mw'], row['logp'], row['molar_refractivity']),
    axis=1
)

# PAINS Filters
pains_results = classification_df['smiles'].apply(check_pains)
classification_df['pains_flag'] = pains_results.apply(lambda x: x[0])
classification_df['pains_alerts'] = pains_results.apply(lambda x: x[1])

print("  ✓ Drug candidacy properties calculated")

Calculating drug candidacy properties...


[16:23:53] Explicit valence for atom # 10 C, 4, is greater than permitted
[16:23:54] Explicit valence for atom # 10 C, 4, is greater than permitted
[16:23:54] Explicit valence for atom # 1 N, 4, is greater than permitted
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] Explicit valence for atom # 6 N, 4, is greater than permitted
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] Explicit valence for atom # 6 N, 4, is greater than permitted
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom w

  ✓ Drug candidacy properties calculated


[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors
[16:23:54] WARNING: not removing hydrogen atom without neighbors


In [12]:
# Calculate physicochemical bins
print("Calculating physicochemical property bins...")

classification_df['polarity_bin'] = classification_df['tpsa'].apply(get_polarity_bin)
classification_df['lipophilicity_bin'] = classification_df['logp'].apply(get_lipophilicity_bin)
classification_df['size_bin'] = classification_df['mw'].apply(get_size_bin)

print("  ✓ Physicochemical bins calculated")

Calculating physicochemical property bins...
  ✓ Physicochemical bins calculated


In [11]:
# Calculate structural properties
print("Calculating structural properties...")

classification_df['heterocycle_present'] = df['nHeteroRing'] > 0
classification_df['aromatic'] = classification_df['smiles'].apply(is_aromatic)
classification_df['peptide_like'] = classification_df['smiles'].apply(check_peptide_like)
classification_df['lipid_like'] = classification_df['smiles'].apply(check_lipid_like)

print("  ✓ Structural properties calculated")

Calculating structural properties...


[16:23:30] Explicit valence for atom # 10 C, 4, is greater than permitted
[16:23:30] Explicit valence for atom # 10 C, 4, is greater than permitted
[16:23:30] Explicit valence for atom # 1 N, 4, is greater than permitted
[16:23:30] WARNING: not removing hydrogen atom without neighbors
[16:23:30] Explicit valence for atom # 6 N, 4, is greater than permitted
[16:23:30] WARNING: not removing hydrogen atom without neighbors
[16:23:30] WARNING: not removing hydrogen atom without neighbors
[16:23:30] WARNING: not removing hydrogen atom without neighbors
[16:23:30] WARNING: not removing hydrogen atom without neighbors
[16:23:30] WARNING: not removing hydrogen atom without neighbors
[16:23:30] WARNING: not removing hydrogen atom without neighbors
[16:23:30] Explicit valence for atom # 6 N, 4, is greater than permitted
[16:23:30] WARNING: not removing hydrogen atom without neighbors
[16:23:30] WARNING: not removing hydrogen atom without neighbors
[16:23:30] WARNING: not removing hydrogen atom w

  ✓ Structural properties calculated


In [10]:
# Prepare the classification dataset
# Map column names from the padel output
classification_df = pd.DataFrame()

# Basic identifiers
classification_df['id'] = range(len(df))
classification_df['name'] = df['Original_Name'].fillna('')
classification_df['smiles'] = df['Original_SMILES'].fillna('')

# Key molecular properties - rename to standardized names
classification_df['tpsa'] = df['TPSA']
classification_df['logp'] = df['CrippenLogP']  # Using Crippen LogP as it's reliable
classification_df['mw'] = df['MW']
classification_df['hbd'] = df['nHBDon_Lipinski']  # Lipinski definition
classification_df['hba'] = df['nHBAcc_Lipinski']  # Lipinski definition
classification_df['rotatable_bonds'] = df['nRotB']
classification_df['ring_count'] = df['nRing']
classification_df['molar_refractivity'] = df['AMR']

print(f"Initial dataset shape: {classification_df.shape}")
print(f"First few rows prepared")

Initial dataset shape: (9584, 11)
First few rows prepared


In [9]:
import json
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem import Crippen
import pandas as pd

# ==================== CLASSIFICATION FUNCTIONS ====================

def get_polarity_bin(tpsa):
    """Bin TPSA into polarity categories"""
    if pd.isna(tpsa):
        return None
    if tpsa < 30:
        return "apolar"
    elif tpsa < 60:
        return "nonpolar"
    elif tpsa < 100:
        return "moderate"
    else:
        return "polar"

def get_lipophilicity_bin(logp):
    """Bin LogP into lipophilicity categories"""
    if pd.isna(logp):
        return None
    if logp < -1:
        return "hydrophilic"
    elif logp < 1:
        return "balanced"
    elif logp < 3:
        return "moderate"
    else:
        return "lipophilic"

def get_size_bin(mw):
    """Bin MW into size categories"""
    if pd.isna(mw):
        return None
    if mw < 160:
        return "small"
    elif mw < 300:
        return "medium"
    elif mw < 500:
        return "large"
    else:
        return "very_large"

def is_aromatic(smiles):
    """Check if molecule contains aromatic atoms"""
    if pd.isna(smiles):
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return any(atom.GetIsAromatic() for atom in mol.GetAtoms())
    except:
        return None

def check_peptide_like(smiles):
    """Check if molecule is peptide-like (contains amide bonds)"""
    if pd.isna(smiles):
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        # Look for amide bonds (C(=O)N)
        amide_pattern = Chem.MolFromSmarts('[#6][#7]')
        carbonyl_pattern = Chem.MolFromSmarts('[#6](=[#8])[#7]')
        return bool(mol.HasSubstructMatch(carbonyl_pattern))
    except:
        return None

def check_lipid_like(smiles):
    """Check if molecule is lipid-like (long chains, many carbons)"""
    if pd.isna(smiles):
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        # Lipid-like: many carbons (>15), typically aliphatic chains
        num_c = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6)
        num_o = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 8)
        # Fatty acids or lipid-like if high C:O ratio and many carbons
        return num_c > 15 and (num_o <= num_c / 3)
    except:
        return None

def check_lipinski(mw, logp, hbd, hba):
    """Check Lipinski's Rule of Five"""
    violations = 0
    if pd.isna(mw) or pd.isna(logp) or pd.isna(hbd) or pd.isna(hba):
        return None, None
    
    if mw > 500:
        violations += 1
    if logp > 5:
        violations += 1
    if hbd > 5:
        violations += 1
    if hba > 10:
        violations += 1
    
    return violations == 0, violations

def check_veber(tpsa, rotatable_bonds):
    """Check Veber's Rule"""
    if pd.isna(tpsa) or pd.isna(rotatable_bonds):
        return None
    return (tpsa <= 140) and (rotatable_bonds <= 10)

def check_egan(logp, tpsa):
    """Check Egan Rule"""
    if pd.isna(logp) or pd.isna(tpsa):
        return None
    return (logp <= 5.88) and (tpsa <= 131)

def check_ghose(mw, logp, molar_refractivity):
    """Check Ghose Filter"""
    if pd.isna(mw) or pd.isna(logp) or pd.isna(molar_refractivity):
        return None
    mw_pass = 160 <= mw <= 480
    logp_pass = -0.4 <= logp <= 5.6
    mr_pass = 40 <= molar_refractivity <= 130
    return mw_pass and logp_pass and mr_pass

def check_pains(smiles):
    """Check PAINS filters - flag problematic substructures"""
    if pd.isna(smiles):
        return None, []
    
    # Common PAINS substructures
    pains_patterns = {
        'thiazole': '[#16]~[#6]~[#7]~[#6]~[#16]',  # Thiazole rings
        'catechol': '[#6]~[#6]([#8])[#6]~[#8]',  # Catechol-like
        'quinone': '[#6](=[#8])[#6]~[#6](=[#8])',  # Quinone
        'Michael_acceptor': '[#6](=[#8])[#6]=[#6]',  # α,β-unsaturated carbonyl
        'cyanamide': '[#7][#6]#[#7]',  # Cyanamide
        'acrylamide': '[#6]=[#6][#6](=[#8])[#7]',  # Acrylamide
    }
    
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None, []
        
        alerts = []
        for name, smarts in pains_patterns.items():
            pattern = Chem.MolFromSmarts(smarts)
            if pattern and mol.HasSubstructMatch(pattern):
                alerts.append(name)
        
        return len(alerts) > 0, alerts
    except:
        return None, []

print("Classification functions loaded successfully!")

Classification functions loaded successfully!


In [8]:
# Install RDKit for chemistry calculations
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"])

0

In [7]:
# Check for key columns we need
key_cols = ['TPSA', 'LogP', 'MW', 'HBD', 'HBA', 'nRotB', 'nRing', 'nHetero']
available = []
for col in key_cols:
    matches = [c for c in df.columns if col.lower() in c.lower()]
    available.extend(matches)

print("Available descriptor columns:")
for col in sorted(set(available)):
    print(f"  {col}")

# Also check if we have SMILES or InChI
print("\nSMILES/InChI columns:")
for col in df.columns:
    if 'smiles' in col.lower() or 'inchi' in col.lower():
        print(f"  {col}")

Available descriptor columns:
  ALogP
  ALogp2
  AMW
  CrippenLogP
  MLogP
  MW
  MWC10
  MWC2
  MWC3
  MWC4
  MWC5
  MWC6
  MWC7
  MWC8
  MWC9
  SHBa
  SHBd
  SwHBa
  SwHBd
  TPSA
  XLogP
  maxHBa
  maxHBd
  maxwHBa
  maxwHBd
  minHBa
  minHBd
  minwHBa
  minwHBd
  nHBAcc
  nHBAcc2
  nHBAcc3
  nHBAcc_Lipinski
  nHBDon
  nHBDon_Lipinski
  nHBa
  nHBd
  nHeteroRing
  nRing
  nRotB
  nRotBt
  nwHBa
  nwHBd

SMILES/InChI columns:
  Original_SMILES


In [6]:
cols = list(df.columns)
print(f"Number of columns: {len(cols)}")
for i, col in enumerate(cols[:20]):
    print(f"{i}: {col}")

Number of columns: 1878
0: Original_Name
1: Original_SMILES
2: nAcid
3: ALogP
4: ALogp2
5: AMR
6: apol
7: naAromAtom
8: nAromBond
9: nAtom
10: nHeavyAtom
11: nH
12: nB
13: nC
14: nN
15: nO
16: nS
17: nP
18: nF
19: nCl


In [4]:
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nFirst row:\n{df.iloc[0]}")

Shape: (9584, 1878)
Columns: ['Original_Name', 'Original_SMILES', 'nAcid', 'ALogP', 'ALogp2', 'AMR', 'apol', 'naAromAtom', 'nAromBond', 'nAtom', 'nHeavyAtom', 'nH', 'nB', 'nC', 'nN', 'nO', 'nS', 'nP', 'nF', 'nCl', 'nBr', 'nI', 'nX', 'ATS0m', 'ATS1m', 'ATS2m', 'ATS3m', 'ATS4m', 'ATS5m', 'ATS6m', 'ATS7m', 'ATS8m', 'ATS0v', 'ATS1v', 'ATS2v', 'ATS3v', 'ATS4v', 'ATS5v', 'ATS6v', 'ATS7v', 'ATS8v', 'ATS0e', 'ATS1e', 'ATS2e', 'ATS3e', 'ATS4e', 'ATS5e', 'ATS6e', 'ATS7e', 'ATS8e', 'ATS0p', 'ATS1p', 'ATS2p', 'ATS3p', 'ATS4p', 'ATS5p', 'ATS6p', 'ATS7p', 'ATS8p', 'ATS0i', 'ATS1i', 'ATS2i', 'ATS3i', 'ATS4i', 'ATS5i', 'ATS6i', 'ATS7i', 'ATS8i', 'ATS0s', 'ATS1s', 'ATS2s', 'ATS3s', 'ATS4s', 'ATS5s', 'ATS6s', 'ATS7s', 'ATS8s', 'AATS0m', 'AATS1m', 'AATS2m', 'AATS3m', 'AATS4m', 'AATS5m', 'AATS6m', 'AATS7m', 'AATS8m', 'AATS0v', 'AATS1v', 'AATS2v', 'AATS3v', 'AATS4v', 'AATS5v', 'AATS6v', 'AATS7v', 'AATS8v', 'AATS0e', 'AATS1e', 'AATS2e', 'AATS3e', 'AATS4e', 'AATS5e', 'AATS6e', 'AATS7e', 'AATS8e', 'AATS0p', '

In [3]:
# Check the shape and columns of the dataframe
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Shape: (9584, 1878)

Columns:
['Original_Name', 'Original_SMILES', 'nAcid', 'ALogP', 'ALogp2', 'AMR', 'apol', 'naAromAtom', 'nAromBond', 'nAtom', 'nHeavyAtom', 'nH', 'nB', 'nC', 'nN', 'nO', 'nS', 'nP', 'nF', 'nCl', 'nBr', 'nI', 'nX', 'ATS0m', 'ATS1m', 'ATS2m', 'ATS3m', 'ATS4m', 'ATS5m', 'ATS6m', 'ATS7m', 'ATS8m', 'ATS0v', 'ATS1v', 'ATS2v', 'ATS3v', 'ATS4v', 'ATS5v', 'ATS6v', 'ATS7v', 'ATS8v', 'ATS0e', 'ATS1e', 'ATS2e', 'ATS3e', 'ATS4e', 'ATS5e', 'ATS6e', 'ATS7e', 'ATS8e', 'ATS0p', 'ATS1p', 'ATS2p', 'ATS3p', 'ATS4p', 'ATS5p', 'ATS6p', 'ATS7p', 'ATS8p', 'ATS0i', 'ATS1i', 'ATS2i', 'ATS3i', 'ATS4i', 'ATS5i', 'ATS6i', 'ATS7i', 'ATS8i', 'ATS0s', 'ATS1s', 'ATS2s', 'ATS3s', 'ATS4s', 'ATS5s', 'ATS6s', 'ATS7s', 'ATS8s', 'AATS0m', 'AATS1m', 'AATS2m', 'AATS3m', 'AATS4m', 'AATS5m', 'AATS6m', 'AATS7m', 'AATS8m', 'AATS0v', 'AATS1v', 'AATS2v', 'AATS3v', 'AATS4v', 'AATS5v', 'AATS6v', 'AATS7v', 'AATS8v', 'AATS0e', 'AATS1e', 'AATS2e', 'AATS3e', 'AATS4e', 'AATS5e', 'AATS6e', 'AATS7e', 'AATS8e', 'AATS0p', 

,Original_Name,Original_SMILES,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,...,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds,BBB
0,sulphasalazine,O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O,1,-1.6366,2.678460,20.9255,52.325102,18,18,42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,moxalactam,COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...,4,-1.8532,3.434350,84.1596,65.253860,11,11,56,...,0.202821,0.515082,0.311531,0.404004,22.493883,107.951242,249.714723,0.589017,1.230617,0
2,clioquinol,Oc1c(I)cc(Cl)c2cccnc12,0,1.7041,2.903957,22.1264,28.605965,10,11,18,...,0.348308,0.488440,0.490672,0.207540,7.749546,14.229627,23.592414,0.476530,1.186652,0
3,bbcpd11 (cimetidine analog) (y-g13),CCNC(=NCCSCc1ncccc1Br)NC#N,0,1.3081,1.711126,58.1882,43.238688,6,6,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,schembl614298,CN1CC[C@]23c4c5ccc(OC6O[C@H](C(=O)O)[C@@H](O)[...,1,-2.3618,5.578099,88.3588,66.801411,6,6,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
